# Topic 5: Partitioning & Bucketing

> **Phase 3 → Week 5 → Topic 5**

---

## Two Types of Partitioning

There are two completely different concepts both called "partitioning" in Spark:

### 1. In-Memory Partitioning — how data is split across executors
```
DataFrame in memory = N RDD partitions, one per executor slot

repartition(N)  → shuffle data into exactly N new partitions (even distribution)
coalesce(N)     → merge partitions, NO full shuffle (faster, but may be uneven)
```

### 2. Storage Partitioning — how data is organized on disk
```
partitionBy("country") when writing → creates folder per country value

data/
├── country=India/
│   └── part-000.parquet
├── country=USA/
│   └── part-000.parquet
└── country=China/
    └── part-000.parquet

Reading with filter country='India' → Spark skips all other folders (partition pruning)
```

### 3. Bucketing — advanced pre-shuffle for joins and aggregations
```
bucketBy(N, "customer_id") when writing → hash(customer_id) % N → bucket number
Rows with same customer_id always go to the same bucket file

When joining two bucketed tables on customer_id → NO SHUFFLE NEEDED!
```

---

## repartition() vs coalesce() — The Key Rule

```
repartition(N):           coalesce(N):
──────────────────────    ────────────────────────
FULL shuffle              NO full shuffle (merge partitions)
Even distribution         May be uneven
Can increase OR decrease  Can ONLY decrease
Use for: making data even Use for: reducing partitions before write
                          (especially after filter that reduces data)
Cost: expensive           Cost: cheap
```

**Rule: Use `coalesce` to reduce partitions, `repartition` to redistribute evenly or increase.**

---

## Interview Cheat Sheet

**Q: What's the difference between repartition() and coalesce()?**
> `repartition(N)` performs a full shuffle and distributes data evenly into exactly N partitions — can increase or decrease. `coalesce(N)` only merges partitions without a full shuffle — cheaper but may result in uneven partitions, and can only decrease the count. Use `coalesce` when reducing partitions before writing (avoids the shuffle cost), and `repartition` when you need even distribution or need to increase parallelism.

**Q: What is storage partitioning (partitionBy) and how does it improve query performance?**
> `partitionBy("col")` when writing creates a separate folder for each unique value of the column. When later reading with a filter on that column, Spark's partition pruning skips all irrelevant folders — it only reads the relevant partition(s). For a 1TB dataset partitioned by country with 100 countries, querying a single country reads only 10GB instead of 1TB.

**Q: What is data skew and how does partitionBy cause it?**
> If you `partitionBy` a column with a very uneven distribution (e.g., 90% of rows have `country=USA`), the USA partition will be 90% of the data — one huge file. This is partition skew. Prefer `partitionBy` on columns with moderate cardinality (dates, regions, status) and reasonably even distribution.

**Q: What is bucketing and when should you use it?**
> Bucketing writes data into a fixed number of files per partition, organized by a hash of a key column. All rows with the same key value always go to the same bucket. When two tables are bucketed on the same key with the same bucket count, joining them requires NO shuffle — each executor joins matching buckets locally. Use it when you repeatedly join large tables on the same key.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

spark = SparkSession.builder \
    .appName("Week5 - Partitioning") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

orders   = spark.read.csv("/workspace/week4/data/orders.csv",   header=True, inferSchema=True)
customers = spark.read.csv("/workspace/week4/data/customers.csv", header=True, inferSchema=True)
print("Ready")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/13 15:39:15 WARN Utils: Your hostname, kali, resolves to a loopback address: 127.0.1.1; using 10.237.41.61 instead (on interface wlan0)
26/06/13 15:39:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/13 15:39:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/06/13 15:39:19 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /workspace/week4/data/orders.csv.
java.io.FileNotFoundException: File /workspace/week4/data/orders.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/workspace/week4/data/orders.csv. SQLSTATE: 42K03

## Part 1: repartition() vs coalesce()

In [2]:
# Start: how many partitions does our DataFrame have?
print(f"Original partition count: {orders.rdd.getNumPartitions()}")

# repartition — full shuffle, even distribution
df_8 = orders.repartition(8)
print(f"After repartition(8): {df_8.rdd.getNumPartitions()} partitions")

df_2 = orders.repartition(2)
print(f"After repartition(2): {df_2.rdd.getNumPartitions()} partitions")

# coalesce — merge only, no shuffle
df_coalesce = orders.coalesce(2)
print(f"After coalesce(2):     {df_coalesce.rdd.getNumPartitions()} partitions")

# CAUTION: coalesce cannot INCREASE partitions
df_coalesce_up = orders.coalesce(100)  # silently does nothing if current < 100
print(f"coalesce(100) when we have fewer: {df_coalesce_up.rdd.getNumPartitions()} (unchanged)")

NameError: name 'orders' is not defined

In [3]:
# Check data distribution across partitions
def partition_sizes(df, label=""):
    sizes = df.rdd.mapPartitions(lambda p: [sum(1 for _ in p)]).collect()
    print(f"{label}: {len(sizes)} partitions, sizes={sizes}, total={sum(sizes)}")

partition_sizes(orders,           "Original")
partition_sizes(orders.repartition(4), "repartition(4) — even")
partition_sizes(orders.coalesce(2),    "coalesce(2) — may be uneven")

NameError: name 'orders' is not defined

In [4]:
# repartition by COLUMN — route specific values to specific partitions
# Rows with same customer_id always go to the same partition
by_customer = orders.repartition(4, "customer_id")
print(f"Repartitioned by customer_id: {by_customer.rdd.getNumPartitions()} partitions")
print("Rows with same customer_id are in the same partition — efficient for per-customer ops")

# Check distribution
partition_sizes(by_customer, "repartition(4, customer_id)")

NameError: name 'orders' is not defined

In [5]:
# Performance: repartition vs coalesce
large = spark.range(1_000_000).withColumn("val", F.rand())

t0 = time.time()
large.repartition(2).count()
repartition_time = time.time() - t0

t0 = time.time()
large.coalesce(2).count()
coalesce_time = time.time() - t0

print(f"repartition(2): {repartition_time:.3f}s  (full shuffle)")
print(f"coalesce(2):    {coalesce_time:.3f}s  (no shuffle — faster)")

repartition(2): 2.373s  (full shuffle)
coalesce(2):    0.253s  (no shuffle — faster)


## Part 2: Storage Partitioning with partitionBy

In [6]:
import subprocess

# Write with partitionBy — creates folder hierarchy
enriched = orders.join(customers, on="customer_id", how="left")

enriched.write.mode("overwrite") \
         .partitionBy("country") \
         .parquet("/tmp/orders_by_country")

print("Written! Directory structure:")
result = subprocess.run(["find", "/tmp/orders_by_country", "-name", "*.parquet"],
                       capture_output=True, text=True)
for line in sorted(result.stdout.strip().split("\n")):
    print(" ", line)

NameError: name 'orders' is not defined

In [7]:
# Partition pruning — reading only relevant folders
partitioned_df = spark.read.parquet("/tmp/orders_by_country")

# Filter on partition column — Spark reads ONLY the India folder
india_only = partitioned_df.filter(F.col("country") == "India")

print("Filter on partition column — check plan for PartitionFilters:")
india_only.explain("formatted")
print()
india_only.show()

# Filter on NON-partition column — must read all folders
high_amount = partitioned_df.filter(F.col("amount") > 500)
print("Filter on non-partition column — must read all folders:")
high_amount.explain("formatted")

26/06/13 15:39:22 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /tmp/orders_by_country.
java.io.FileNotFoundException: File /tmp/orders_by_country does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.R

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/tmp/orders_by_country. SQLSTATE: 42K03

In [8]:
# Multi-column partitioning — hierarchical folders
# Common pattern: partition by year/month for time-series data
orders_with_date = orders.withColumn("year",  F.year(F.to_date("order_date"))) \
                          .withColumn("month", F.month(F.to_date("order_date")))

orders_with_date.write.mode("overwrite") \
                .partitionBy("year", "month") \
                .parquet("/tmp/orders_by_yearmonth")

print("Year/month hierarchy:")
result2 = subprocess.run(["find", "/tmp/orders_by_yearmonth", "-type", "f"],
                        capture_output=True, text=True)
for line in sorted(result2.stdout.strip().split("\n")):
    print(" ", line)

NameError: name 'orders' is not defined

In [9]:
# Partition count problem — too many small files
# Spark creates one file per partition per partition value
# Solution: repartition(1) before write, OR coalesce()

# Bad: creates many small files
# orders.repartition(10).partitionBy("country").parquet(...)  → 10 * N_countries files!

# Good: repartition by the partition column → 1 file per country value
orders.join(customers, on="customer_id", how="left") \
      .repartition("country") \
      .write.mode("overwrite") \
      .partitionBy("country") \
      .parquet("/tmp/orders_optimized")

print("Optimized write (repartition by partition column first):")
result3 = subprocess.run(["find", "/tmp/orders_optimized", "-name", "*.parquet"],
                        capture_output=True, text=True)
files = [l for l in result3.stdout.strip().split("\n") if l]
print(f"  {len(files)} files (1 per country)")  
for f in sorted(files):
    print(" ", f)

NameError: name 'orders' is not defined

## Part 3: Bucketing

In [10]:
# Bucketing requires saving as a managed table (not just files)
# Spark needs to remember the bucketing metadata in the catalog


# Write orders and customers as bucketed tables
orders.write.mode("overwrite") \
       .bucketBy(4, "customer_id") \
       .sortBy("customer_id") \
       .saveAsTable("orders_bucketed")

customers.write.mode("overwrite") \
          .bucketBy(4, "customer_id") \
          .sortBy("customer_id") \
          .saveAsTable("customers_bucketed")

print("Bucketed tables saved!")

NameError: name 'orders' is not defined

In [11]:
# Read bucketed tables
orders_b    = spark.table("orders_bucketed")
customers_b = spark.table("customers_bucketed")

# Join — Spark detects both are bucketed on customer_id with same bucket count
# → NO shuffle needed!
bucketed_join = orders_b.join(customers_b, on="customer_id", how="inner")

print("Join of two bucketed tables — look for absence of Exchange (shuffle):")
bucketed_join.explain()

# Compare with non-bucketed join
print("\nNon-bucketed join — shows Exchange (shuffle):")
orders.join(customers, on="customer_id", how="inner").explain()

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `orders_bucketed` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01;
'UnresolvedRelation [orders_bucketed], [], false


In [12]:
# Optimal partition size rule
print("""
Optimal Partition Sizing Guidelines:
─────────────────────────────────────────────────────────────────────
Target partition size: 128MB - 256MB (Parquet), 64MB - 128MB (CSV/JSON)

Formula: num_partitions = total_data_size_mb / target_partition_size_mb

Example: 10GB Parquet data, target 200MB per partition
         = 10000 / 200 = 50 partitions

spark.sql.shuffle.partitions (default 200):
  - Too high: many tiny partitions → task scheduling overhead
  - Too low:  few huge partitions → memory pressure, slow stragglers
  - AQE auto-coalesces these at runtime (Spark 3.0+)

Rule of thumb:
  - 2-4 partitions per CPU core for compute-heavy operations
  - Set spark.sql.shuffle.partitions to 2x your cluster core count
  - Let AQE handle the fine-tuning at runtime
""")


Optimal Partition Sizing Guidelines:
─────────────────────────────────────────────────────────────────────
Target partition size: 128MB - 256MB (Parquet), 64MB - 128MB (CSV/JSON)

Formula: num_partitions = total_data_size_mb / target_partition_size_mb

Example: 10GB Parquet data, target 200MB per partition
         = 10000 / 200 = 50 partitions

spark.sql.shuffle.partitions (default 200):
  - Too high: many tiny partitions → task scheduling overhead
  - Too low:  few huge partitions → memory pressure, slow stragglers
  - AQE auto-coalesces these at runtime (Spark 3.0+)

Rule of thumb:
  - 2-4 partitions per CPU core for compute-heavy operations
  - Set spark.sql.shuffle.partitions to 2x your cluster core count
  - Let AQE handle the fine-tuning at runtime



## Part 4: Putting It All Together — Optimized Write Pattern

In [13]:
# Production write pattern for a large fact table
# 1. Join/transform
# 2. Repartition by partition column (ensures 1 file per partition value)
# 3. Write with partitionBy + Parquet

fact_orders = (
    orders
    .withColumn("year",  F.year(F.to_date("order_date")))
    .withColumn("month", F.month(F.to_date("order_date")))
    .withColumn("order_date", F.to_date("order_date"))
)

# Write: partition by year/month, sorted by order_date within
fact_orders \
    .repartition("year", "month") \
    .write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .option("compression", "snappy") \
    .parquet("/tmp/fact_orders")

# Read with partition pruning
spark.read.parquet("/tmp/fact_orders") \
     .filter((F.col("year") == 2023) & (F.col("month") == 6)) \
     .show()

print("Only year=2023/month=6 folder was read — partition pruning in action.")

NameError: name 'orders' is not defined

## Exercises

1. Create a DataFrame of 10M rows with a `region` column (5 unique values). Write it with `partitionBy("region")`. Read it back and filter for one region — confirm partition pruning in the explain plan.
2. What is the difference between `repartition(10)` and `repartition(10, "customer_id")`? When would you use the second form?
3. After a filter that keeps only 5% of rows, should you use `coalesce()` or `repartition()` before writing? Why?
4. You have 1TB of sales data partitioned by `date` (daily). A common query is `WHERE date BETWEEN '2023-01-01' AND '2023-01-31'`. Is this a good partition column choice? What about `user_id`?
5. Explain what happens when you `partitionBy` on a column with 100,000 unique values (like user_id). Why is this dangerous?

In [14]:
# Exercise 1: Partition pruning demo
large_df = spark.range(100_000) \
    .withColumn("region", (F.col("id") % 5).cast("string")) \
    .withColumn("value", F.rand() * 1000)

large_df.write.mode("overwrite") \
         .partitionBy("region") \
         .parquet("/tmp/large_partitioned")

back = spark.read.parquet("/tmp/large_partitioned")
print("Reading with region='2' filter — partition pruning:")
back.filter(F.col("region") == "2").explain("formatted")

Reading with region='2' filter — partition pruning:
== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [id#18L, value#19, region#20]
Batched: true
Location: InMemoryFileIndex [file:/tmp/large_partitioned]
PartitionFilters: [isnotnull(region#20), (region#20 = 2)]
ReadSchema: struct<id:bigint,value:double>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [id#18L, value#19, region#20]




In [15]:
# Exercise 3: coalesce after filter
# Original: 100k rows in 4 partitions
# After filter: ~5k rows — 4 tiny partitions
# Before writing: merge to 1-2 partitions

filtered = large_df.filter(F.col("region") == "0")  # ~20% of data
print(f"After filter: {filtered.rdd.getNumPartitions()} partitions ({filtered.count()} rows)")

coalesced = filtered.coalesce(1)  # merge — no shuffle needed, data is already small
print(f"After coalesce(1): {coalesced.rdd.getNumPartitions()} partition")
print("Use coalesce (not repartition) because data already reduced — no need for full shuffle")

After filter: 4 partitions (20000 rows)
After coalesce(1): 1 partition
Use coalesce (not repartition) because data already reduced — no need for full shuffle
